# EFM 1024³ Cosmogenesis Master Validation Pipeline (TPU v6e - Periodic Physics Parity)
========================================================================================
Definitive 1-to-1 executable matching original periodic baseline (`Copy_of_FULLNUF.ipynb`).
Uses Periodic torch.roll Boundary Conditions (0 boundary energy loss) at 1024³ resolution.

In [ ]:
import os
import sys
import time
import glob
import shutil
import numpy as np
from tqdm import tqdm
import torch
import torch.nn.functional as F

# 1. MOUNT GOOGLE DRIVE CLOUD FIRST (BEFORE CREATING ANY PATHS)
from google.colab import drive
print("--- Mounting Real Google Drive Cloud Storage ---")
drive.mount('/content/drive', force_remount=False)

# Verify Real Google Drive Connection
if not os.path.exists('/content/drive/MyDrive'):
    raise RuntimeError("CRITICAL ERROR: Google Drive failed to mount at /content/drive/MyDrive!")
else:
    print("=== Real Google Drive Successfully Connected: /content/drive/MyDrive ===")

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    device = torch_xla.device()
    TPU_AVAILABLE = True
    print(f"=== PyTorch-XLA Engine Initialized on TPU: {device} ===")
except Exception as e:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    TPU_AVAILABLE = False
    print(f"=== Running on Standard Fallback Device: {device} ({e}) ===")

TPU_DTYPE = torch.float32

# EXPLICIT REAL GOOGLE DRIVE PATH (ON REAL MOUNTED GOOGLE DRIVE)
GDRIVE_STORAGE_DIR = '/content/drive/MyDrive/EFM_Simulations/data/FirstPrinciples_Dynamic_N1024_v12_PeriodicParity_TPU/'
os.makedirs(GDRIVE_STORAGE_DIR, exist_ok=True)

# LOCAL NVME SCRATCH DIR
LOCAL_NVME_TMP_DIR = '/tmp/EFM_Checkpoints_TPU_Periodic/'
os.makedirs(LOCAL_NVME_TMP_DIR, exist_ok=True)

print(f"[CONFIRMED] Real Google Drive Destination: {GDRIVE_STORAGE_DIR}")
print(f"[CONFIRMED] Local NVMe Temp Destination   : {LOCAL_NVME_TMP_DIR}")


## 1. Physics Configuration & Periodic Stencil Operators (0 Energy Loss)

In [ ]:
# --- Exact EFM Physics Configuration (N=1024 Periodic Parity) ---
config = {
    'N': 1024,
    'L_sim_unit': 40.0,
    'T_steps': 267000,
    'dt_cfl_factor': 0.001,
    'c_sim_unit': 1.0,

    # Epoch Parameters
    'transition_step': 1000,
    'm_sq_inflation': -0.25,

    # Density-Dependent State-Switching
    'rho_threshold': 0.0005,
    'k_density_coupling': 0.01,

    # S/T Vacuum State
    'm_sq_vacuum': 0.01,
    'g_vacuum': 0.1,

    # S=T Matter State
    'm_sq_particle': 1.0,
    'g_particle': -0.1,

    # Universal Terms
    'alpha_structure': 0.7,
    'eta_sim': 0.01,
    'delta_sim': 0.0002,

    # Stability & Boundary
    'sponge_width': 0.1,
    'sponge_strength': 0.99,
    'initial_noise_amplitude': 1.0e-4,

    'history_every_n_steps': 1000,
    'checkpoint_every_n_steps': 10000
}

config['dx_sim_unit'] = config['L_sim_unit'] / config['N']
config['dt_sim_unit'] = config['dt_cfl_factor'] * config['dx_sim_unit'] / config['c_sim_unit']
config['run_id'] = f"TPU_N{config['N']}_T{config['T_steps']}_StructureV12"

# --- PERIODIC STENCIL OPERATORS (TORCH.ROLL - 0 BOUNDARY ENERGY LOSS) ---

def conv_laplacian_tpu_periodic(phi: torch.Tensor, dx: float) -> torch.Tensor:
    dx_sq = dx * dx
    lap = (
        torch.roll(phi, -1, 0) + torch.roll(phi, 1, 0) +
        torch.roll(phi, -1, 1) + torch.roll(phi, 1, 1) +
        torch.roll(phi, -1, 2) + torch.roll(phi, 1, 2) -
        6.0 * phi
    ) / dx_sq
    return lap

def compute_grad_phi_sq_tpu_periodic(phi: torch.Tensor, dx: float) -> torch.Tensor:
    two_dx = 2.0 * dx
    gx = (torch.roll(phi, -1, 0) - torch.roll(phi, 1, 0)) / two_dx
    gy = (torch.roll(phi, -1, 1) - torch.roll(phi, 1, 1)) / two_dx
    gz = (torch.roll(phi, -1, 2) - torch.roll(phi, 1, 2)) / two_dx
    return gx**2 + gy**2 + gz**2

def apply_boundary_damping_tpu_inplace(phi_dot: torch.Tensor, sponge_width_frac: float, sponge_strength: float):
    # Match Copy_of_FULLNUF.ipynb: ramp from 0.99 to 1.0 (subtle 1% buffer, 0 energy destruction)
    N = phi_dot.shape[0]
    w = int(N * sponge_width_frac)
    ramp = torch.linspace(sponge_strength, 1.0, w, device=phi_dot.device, dtype=phi_dot.dtype)
    phi_dot[:w, :, :] *= ramp.view(-1, 1, 1)
    phi_dot[-w:, :, :] *= ramp.view(-1, 1, 1).flip(0)
    phi_dot[:, :w, :] *= ramp.view(1, -1, 1)
    phi_dot[:, -w:, :] *= ramp.view(1, -1, 1).flip(1)
    phi_dot[:, :, :w] *= ramp.view(1, 1, -1)
    phi_dot[:, :, -w:] *= ramp.view(1, 1, -1).flip(2)

def nlkg_derivative_tpu(phi: torch.Tensor, phi_dot: torch.Tensor, dx: float, c_sq: float,
                        k_density: float, rho_thresh: float,
                        m_sq_vac: float, g_vac: float,
                        m_sq_part: float, g_part: float,
                        eta: float, alpha: float, delta: float,
                        inflation_w: float) -> tuple[torch.Tensor, torch.Tensor]:
    lap_phi = conv_laplacian_tpu_periodic(phi, dx)

    rho = k_density * torch.pow(phi, 2)
    m_sq_dyn = torch.where(rho > rho_thresh, torch.tensor(m_sq_part, dtype=torch.float32, device=phi.device), torch.tensor(m_sq_vac, dtype=torch.float32, device=phi.device))
    g_dyn = torch.where(rho > rho_thresh, torch.tensor(g_part, dtype=torch.float32, device=phi.device), torch.tensor(g_vac, dtype=torch.float32, device=phi.device))

    w = inflation_w
    m_sq_eff = w * config['m_sq_inflation'] + (1.0 - w) * m_sq_dyn
    g_eff = (1.0 - w) * g_dyn
    eta_eff = (1.0 - w) * eta
    alpha_eff = (1.0 - w) * alpha
    delta_eff = (1.0 - w) * delta

    potential_force = m_sq_eff * phi + g_eff * torch.pow(phi, 3) + eta_eff * torch.pow(phi, 5)
    grad_phi_abs_sq = compute_grad_phi_sq_tpu_periodic(phi, dx)
    alpha_term = alpha_eff * phi * phi_dot * grad_phi_abs_sq
    delta_term = delta_eff * torch.pow(phi_dot, 2) * phi

    phi_ddot = c_sq * lap_phi - potential_force + alpha_term - delta_term
    return phi_dot, phi_ddot

# --- EULER-LEAPFROG SOLVER (FLOAT32) --- 
def update_phi_leapfrog_tpu(phi_current: torch.Tensor, phi_dot_current: torch.Tensor, dt: float, dx: float, c_sq: float,
                            k_density: float, rho_thresh: float, m_sq_vac: float, g_vac: float,
                            m_sq_part: float, g_part: float, eta: float, alpha: float, delta: float,
                            inflation_w: float
                            ) -> tuple[torch.Tensor, torch.Tensor]:
    _, phi_ddot = nlkg_derivative_tpu(
        phi_current, phi_dot_current, dx, c_sq,
        k_density, rho_thresh, m_sq_vac, g_vac,
        m_sq_part, g_part, eta, alpha, delta, inflation_w
    )
    phi_next = phi_current + dt * phi_dot_current
    phi_dot_next = phi_dot_current + dt * phi_ddot
    return phi_next, phi_dot_next

print("=== Float32 Periodic 1024³ TPU Solver Compiled Successfully ===")

## 2. Multi-State HDS Census Function

In [ ]:
def run_multi_state_hds_census(checkpoint_path: str):
    print("\n========================================================================================")
    print(f"   MULTI-STATE HDS PHASE SPACE CENSUS (Checkpoint: {os.path.basename(checkpoint_path)})")
    print("========================================================================================")
    try:
        data = np.load(checkpoint_path, allow_pickle=True)
        phi = data['phi_cpu'].astype(np.float32) if 'phi_cpu' in data else data['phi_final_cpu'].astype(np.float32)
        t_step = int(data['t_step']) if 't_step' in data else 0
    except Exception as e:
        print(f"Error loading checkpoint: {e}")
        return

    k = config['k_density_coupling']
    rho = k * (phi ** 2)

    stat_matter_thresh = np.percentile(rho, 99.99)
    stat_quantum_thresh = np.percentile(rho, 99.999)

    s_t_mask = rho < stat_matter_thresh
    s_eq_t_mask = (rho >= stat_matter_thresh) & (rho < stat_quantum_thresh)
    t_s_mask = rho >= stat_quantum_thresh

    s_t_vol = (np.sum(s_t_mask) / rho.size) * 100.0
    s_eq_t_vol = (np.sum(s_eq_t_mask) / rho.size) * 100.0
    t_s_vol = (np.sum(t_s_mask) / rho.size) * 100.0

    print(f"  * Step {t_step} Multi-State HDS Census:")
    print(f"    - S/T (Cosmic Vacuum State) Volume : {s_t_vol:.4f}%")
    print(f"    - S=T (Matter State) Volume        : {s_eq_t_vol:.4f}%")
    print(f"    - T/S (Quantum State) Volume       : {t_s_vol:.4f}%")
    print(f"  * 99.99th Percentile Density Threshold: {stat_matter_thresh:.6e}")
    print("========================================================================================\n")

## 3. Main Simulation Loop

In [ ]:
if __name__ == '__main__':
    print("--- INITIATING EFM 1024³ COSMOGENESIS SIMULATION (TPU v6e Periodic Float32 Run) ---")
    print(f"Real Google Drive Storage Directory: {GDRIVE_STORAGE_DIR}")
    print(f"Execution Device: {device} | Precision: {TPU_DTYPE}")

    # --- AUTO-SYNC ANY STUCK LOCAL CHECKPOINTS DIRECTLY TO REAL GOOGLE DRIVE ---
    local_checkpoints = glob.glob(os.path.join(LOCAL_NVME_TMP_DIR, "CHECKPOINT_step_*.npz"))
    for local_f in local_checkpoints:
        gdrive_target_f = os.path.join(GDRIVE_STORAGE_DIR, os.path.basename(local_f))
        print(f"[COPYING LOCAL CHECKPOINT TO GOOGLE DRIVE] {os.path.basename(local_f)} -> {gdrive_target_f}")
        shutil.copyfile(local_f, gdrive_target_f)
        print(f"[SUCCESS] Copied {os.path.basename(local_f)} to Real Google Drive!")
    # --------------------------------------------------------------------------

    start_step = 0
    phi_current = None
    phi_dot_current = None

    checkpoint_files = glob.glob(os.path.join(GDRIVE_STORAGE_DIR, "CHECKPOINT_step_*.npz")) + \
                       glob.glob(os.path.join(LOCAL_NVME_TMP_DIR, "CHECKPOINT_step_*.npz"))
    if checkpoint_files:
        def get_step_from_filename(f):
            try: return int(os.path.basename(f).split('_')[2])
            except: return -1
        latest_checkpoint = max(checkpoint_files, key=get_step_from_filename)
        print(f"--- Found latest checkpoint: {os.path.basename(latest_checkpoint)} ---")
        try:
            with np.load(latest_checkpoint, allow_pickle=True) as data:
                phi_current = torch.from_numpy(data['phi_cpu'].astype(np.float32)).to(device, dtype=TPU_DTYPE)
                phi_dot_current = torch.from_numpy(data['phi_dot_cpu'].astype(np.float32)).to(device, dtype=TPU_DTYPE)
                start_step = int(data['t_step'])
                print(f"--- Resuming simulation directly from step {start_step} ---")
        except Exception as e:
            print(f"Could not load checkpoint: {e}. Initializing fresh universe.")
            start_step = 0; phi_current = None; phi_dot_current = None

    if phi_current is None:
        print("--- Initializing universe from random noise (Float32). ---")
        torch.manual_seed(42)
        phi_current = (torch.rand(config['N'], config['N'], config['N'], device=device, dtype=TPU_DTYPE) * config['initial_noise_amplitude'])
        phi_dot_current = torch.zeros_like(phi_current, dtype=TPU_DTYPE)

    history_size = config['T_steps'] // config['history_every_n_steps']
    max_phi_history = np.zeros(history_size)
    history_idx = start_step // config['history_every_n_steps']

    pbar = tqdm(range(start_step, config['T_steps']), desc=f"TPU Structure Formation ({config['N']}³)", initial=start_step, total=config['T_steps'])
    sim_start_time = time.time()
    dt, dx, c_sq = config['dt_sim_unit'], config['dx_sim_unit'], config['c_sim_unit']**2

    for t_step in pbar:
        infl_w = 1.0 if (t_step < config['transition_step']) else 0.0
        if t_step == config['transition_step']:
            print("\n--- Reheating Transition: Activating Density-Dependent Physics on TPU ---")

        phi_next, phi_dot_next = update_phi_leapfrog_tpu(
            phi_current, phi_dot_current, dt, dx, c_sq,
            config['k_density_coupling'], config['rho_threshold'],
            config['m_sq_vacuum'], config['g_vacuum'],
            config['m_sq_particle'], config['g_particle'],
            config['eta_sim'], config['alpha_structure'], config['delta_sim'],
            inflation_w=infl_w
        )

        apply_boundary_damping_tpu_inplace(phi_dot_next, config['sponge_width'], config['sponge_strength'])
        phi_current, phi_dot_current = phi_next, phi_dot_next

        if TPU_AVAILABLE:
            torch_xla.sync()

        if (t_step + 1) % config['history_every_n_steps'] == 0:
            if torch.any(torch.isinf(phi_current)) or torch.any(torch.isnan(phi_current)):
                print(f"\nERROR: NaN/Inf detected at step {t_step + 1}! Halting."); break
            max_phi = torch.max(torch.abs(phi_current)).item()
            if history_idx < history_size: max_phi_history[history_idx] = max_phi; history_idx += 1
            epoch_str = 'Inflation' if t_step < config['transition_step'] else 'Structure'
            pbar.set_postfix({'Max|φ|': f'{max_phi:.3e}', 'Epoch': epoch_str})

        # --- DIRECT GDRIVE COPY CHECKPOINTING ---
        if (t_step + 1) % config['checkpoint_every_n_steps'] == 0:
            ckpt_name = f"CHECKPOINT_step_{t_step+1}_{config['run_id']}.npz"
            local_ckpt_path = os.path.join(LOCAL_NVME_TMP_DIR, ckpt_name)
            gdrive_ckpt_path = os.path.join(GDRIVE_STORAGE_DIR, ckpt_name)

            if TPU_AVAILABLE: torch_xla.sync(wait=True)

            phi_host = phi_current.cpu().numpy().astype(np.float16)
            phi_dot_host = phi_dot_current.cpu().numpy().astype(np.float16)

            np.savez_compressed(
                local_ckpt_path,
                phi_cpu=phi_host,
                phi_dot_cpu=phi_dot_host,
                t_step=t_step + 1,
                config=config,
                max_phi_history=max_phi_history
            )
            # COPY DIRECTLY TO REAL MOUNTED GOOGLE DRIVE FOLDER
            shutil.copyfile(local_ckpt_path, gdrive_ckpt_path)
            print(f"\n--- Checkpoint committed directly to Real Google Drive: {gdrive_ckpt_path} ---")
            run_multi_state_hds_census(local_ckpt_path)
